# Welcome to Snowflake Notebooks in Workspaces! 🌟
Snowflake Notebooks are fully-managed Jupyter-powered notebook built for end-to-end DS and ML development on Snowflake data.

This includes:
- 🐍 **Familiar Jupyter experience** - Directly connected to the governed Snowflake data.
- 💻 **Inside Snowflake Workspaces** - Organize and run notebooks alongside your other files.
- 🧠 **Powerful for AI/ML** - Runs in our pre-built Container Runtime (CPU or GPU) with popular DS/ML packages pre-installed.
- 🏛️ **Governed collaboration** - Enable multiple users to collaborate via Git-backed Workspaces or Shared Workspaces.

## 1. Connect and run the notebook
Press [Run all] or [Connect] to create and connect your notebook to a service.

Optionally, run the below Python cell to inspect all pre-installed packages. To use a package, run `import <package_name>`. See example in "4. Visualize customer patterns". Learn more about [Managing packages and runtime](https://docs.snowflake.com/en/user-guide/ui-snowsight/notebooks-in-workspaces/notebooks-in-workspaces-packages-runtime)

In [ ]:
!pip freeze # Shows all pre-installed packages

# Congrats! Now create your own! ✅
Click "Create file" in a workspace and select Notebooks.

**Resources to help you get started**
- [YouTube Video Overview](https://www.youtube.com/watch?v=_kFhFIvnIrQ)
- [ML Quickstart using GPUs](https://www.snowflake.com/en/developers/guides/accelerate-topic-modeling-with-gpus-in-snowflake-ml/) - just download the .ipynb and upload to your Workspace!
- [Documentation](https://docs.snowflake.com/en/user-guide/ui-snowsight/notebooks-in-workspaces/notebooks-in-workspaces-limitations)

# Snowpark ML Modeling — Notebook Overview

This notebook is loosely based on the **“Snowpark ML Modeling – Part I”** demo from the Coursera course **“Intro to Snowflake for Devs, Data Scientists, Data Engineers.”**  
The original material was developed in a Jupyter notebook; this version adapts it for use inside a **Snowflake Workspace notebook**.

---

## 🎯 Goal
Evaluate how effectively **XGBoost** can make predictions in a simple ML scenario.

---

## 📘 Scenario
Use **synthetic location data** to predict where an **ice cream truck** will appear.

---

## 📘 Synthesized Data

To train the model, we generate **20 years of synthetic daily truck‑location data** based on a simple, rule‑driven schedule:

### **January Pattern**
- Visits **Neighborhood 1** on days: 1, 8, 15, 22, 29  
- Visits **Neighborhood 2** on *all other days*  
- (Reason: the driver’s mother lives in Neighborhood 1 in January)

### **February–November Pattern**
- Day 1 → Neighborhood 1  
- Day 2 → Neighborhood 2  
- Day 3 → Neighborhood 3  
- …continuing up to Neighborhood 7  
- After day 7, the pattern loops:
  - Day 8 → Neighborhood 1  
  - Day 9 → Neighborhood 2  
  - etc.  
- Pattern resets at the start of each month

### **December Pattern**
- Visits **Neighborhood 8 every day**  
- (Reason: the driver loves the holiday decorations there)

### **Dataset Construction**
- Generate one full year of daily records using the rules above  
- Repeat (concatenate) the dataset **20 times**  
- Upload the resulting dataset to Snowflake via Snowsight

---

## ⚠️ Known Limitations
1. The **ipywidgets** library is *not* available in Workspace notebooks.



In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()
session.sql("SELECT CURRENT_TIMESTAMP()").show()

In [ ]:
# pip install -U ipywidgets


In [ ]:
# from snowflake.snowpark import Session
from snowflake.ml.modeling.xgboost import XGBClassifier
from snowflake.snowpark.functions import col
# from snowflake.ml.modeling import preprocessing

In [ ]:
# Here’s the neighborhood visiting pattern the truck follows:
# In January, the truck goes to N1 on the 1st, 8th, 15th, 22nd, and 29th, and N2 the other days.

# From February through November, it goes to:
# N1 on the 1st
# N2 on the 2nd
# N3 on the 3rd
# N4 on the 4th
# N5 on the 5th
# N6 on the 6th
# N7 on the 7th
# N1 on the 8th
# N2 on the 9th
# etc.

# Every December, it only goes to N8.

month_days = [31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]

pre = {}

for i,month_length in enumerate(month_days):
    month = i + 1

    for day in range(1,month_length+1):
        
        # In January, it goes to neighborhood 1 on Mondays, and neighborhood 2 the other days.
        if ((month) == 1):
            if (day) % 7 == 1:
                pre[(month,day)] = 1
            else:
                pre[(month,day)] = 2
                
        # From February through November, it goes to neighborhood 1 on the 1st, 2 on the 2nd, 3 on the 3rd,
        # 4 on the 4th, 5 on the 5th, 6 on the 6th, and 7 on the 7th, 1 on the 8th, 2 on the 9th, etc.
        elif ((month) <= 11):
            pre[(month,day)] = ((day-1) % 7) + 1

        # Every December, it only goes to neighborhood 8.
        elif ((month) == 12):
            pre[(month,day)] = 8

# see what the pre dictionary looks like
pre



In [ ]:
from snowflake.snowpark.types import StructType, StructField, IntegerType

session.sql("USE DATABASE NORTHSTAR_ML").collect()
session.sql("USE SCHEMA ML").collect()

schema = StructType([StructField("MONTH", IntegerType()), StructField("DAY", IntegerType()), StructField("NEIGHBORHOOD", IntegerType())])
rows = [(k[0], k[1], v) for k, v in pre.items()]
snowpark_df = session.create_dataframe(rows, schema=schema)
snowpark_df.write.mode("overwrite").save_as_table("NORTHSTAR_ML.ML.DF_CLEAN")

In [ ]:
snowpark_df.show(40)

In [ ]:
snowpark_df.count()

In [ ]:
snowpark_df.describe().show()

In [ ]:
snowpark_df.group_by("NEIGHBORHOOD").count().show()

## Snowpark ML Modeling with XGBClassifier

### What is Snowpark ML Modeling?

Snowpark ML incorporates many of the most important functions and methods from popular open-source Python ML libraries — including **scikit-learn**, **XGBoost**, and **LightGBM** — so they're easy to use directly in Snowflake.

A key benefit: when you train a model in Snowflake, training can happen in a **distributed way automatically**. Snowflake splits your data and trains on different parts simultaneously. This parallelization can significantly speed up the training process.

### What Does XGBClassifier Do?

We're using `XGBClassifier` from `snowflake.ml.modeling.xgboost`. At a high level, this model tries to figure out the relationship between:

- **Labels (output):** Neighborhood 1 through 8
- **Input variables (features):** Day and Month

The goal: given some input data, predict what the output neighborhood will be.

### How Tree-Based Models Work

Think of it like the game **20 Questions** — the model comes up with a series of clever yes/no questions about your data that progressively bin it into buckets of similar items.

For example:
1. *Do some neighborhoods appear more in the second half of the year?* — Yes, because neighborhood 8 only appears in December.
2. *Do some neighborhoods appear more in December specifically?* — This is a jackpot split, cleanly isolating neighborhood 8.

The model keeps asking these questions, splitting the data into smaller and smaller buckets that become more concentrated with a single label. Drawing out this logic looks like branches hanging down — hence the name **tree-based models**.

### Label Adjustment: 1-8 to 0-7

`XGBClassifier` requires labels to start from **0**, not 1. Since our neighborhoods are numbered 1-8, we need to transform them to **0-7** before training. Two approaches for this transformation are shown below.

In [ ]:
from snowflake.ml.modeling.preprocessing import LabelEncoder

# Option 1: One way to scale your target (neighborhood) so you can use it in the XGBClassifier model
test = snowpark_df.withColumn('NEIGHBORHOOD2', snowpark_df.neighborhood - 1).drop("Neighborhood")
test.show()
# Option 2: now use scikit-learns LabelEncoder -- a more general solution -- through Snowpark ML 
le = LabelEncoder(input_cols=['NEIGHBORHOOD'], output_cols= ['NEIGHBORHOOD2'], drop_input_cols=True)

# apply the LabelEncoder
fitted = le.fit(snowpark_df.select("NEIGHBORHOOD"))

snowpark_df_prepared = fitted.transform(snowpark_df)

snowpark_df_prepared.show()

In [ ]:
# split the data into a training set and a test set
train_snowpark_df, test_snowpark_df = snowpark_df_prepared.randomSplit([0.9, 0.1])
# save training data
train_snowpark_df.write.mode("overwrite").save_as_table("NORTHSTAR_ML.ML.DF_CLEAN_TRAIN")

# save test data
test_snowpark_df.write.mode("overwrite").save_as_table("NORTHSTAR_ML.ML.DF_CLEAN_TEST")


In [ ]:
count_train = session.sql("select count(*) from NORTHSTAR_ML.ML.DF_CLEAN_TRAIN").collect()[0][0]
count_test = session.sql("select count(*) from NORTHSTAR_ML.ML.DF_CLEAN_TEST").collect()[0][0]
print(f'Training rows = {count_train}')
print(f'Test rows = {count_test}')
# train_snowpark_df.show()

In [ ]:
# create and train the XGBClassifier model
FEATURE_COLS = ["MONTH", "DAY"]
LABEL_COLS = ["NEIGHBORHOOD2"]

# Train an XGBoost model on snowflake.
xgboost_model = XGBClassifier(
    input_cols=FEATURE_COLS,
    label_cols=LABEL_COLS
)

xgboost_model.fit(train_snowpark_df)


In [ ]:
# check the accuracy using scikit-learns score functionality through Snowpark ML
accuracy = xgboost_model.score(test_snowpark_df)

print("Accuracy: %.2f%%" % (accuracy * 100.0))

## The Instructor’s Note  
Before recapping the workflow, the instructor pointed out that everything we did in Snowpark ML could also have been done inside a SQL workflow using the Snowflake Cortex ML `CLASSIFICATION` function. However, the lesson focused on Snowpark ML because:

- We had already explored Snowflake Cortex LLM functions and understood how Cortex works.
- Snowpark ML offers more customization and flexibility than Cortex ML functions.
- It’s valuable to know both approaches so you can choose the right tool depending on how much control your ML workflow requires.

### Notes  
> **Note 1:** The instructor demonstrated this workflow using a *local Jupyter notebook*.  
> I followed the same steps using a **Snowflake Workspace Notebook**, which behaves similarly but runs directly inside Snowflake. The concepts and code are identical; only the environment differs.

> **Note 2:** The Python cells that follow this Markdown show an **equivalent solution implemented using Snowflake Cortex ML functions**, as the instructor mentioned. I asked Cortex to generate that version, and the resulting workflow appears to run correctly.

---

## Summary  
We learned how to:

1. **Connect a local or Workspace notebook to Snowflake**  
   - Created a Snowpark session using `Session.builder.configs().create()`.

2. **Load a Snowflake table into a Snowpark DataFrame**  
   - Used `session.table("<table_name>")` to access data.

3. **Explore and transform data with Snowpark DataFrame methods**  
   - Applied operations such as `count()`, `describe()`, `group_by()`, and `with_column()`.

4. **Preprocess data using Snowpark ML**  
   - Encoded categorical labels with `LabelEncoder`.

5. **Split data into training and test sets**  
   - Used `random_split()` to create training and test DataFrames.

6. **Write processed data back to Snowflake**  
   - Saved a DataFrame as a table using `write.mode("overwrite").save_as_table()`.

7. **Train a machine learning model**  
   - Fit an `XGBClassifier` on the training data.

8. **Evaluate the model**  
   - Calculated accuracy using the classifier’s `.score()` method.

This covered a full end‑to‑end ML workflow in Snowpark ML — a lot of ground, especially if you’re new to machine learning.

---

I asked Cortex to show me how to do that, and it came up with the solution below.


In [ ]:
%%sql -r dataframe_2
CREATE OR REPLACE VIEW NORTHSTAR_ML.ML.CLASSIFICATION_TRAIN_VIEW AS
  SELECT MONTH, DAY, NEIGHBORHOOD
  FROM NORTHSTAR_ML.ML.DF_CLEAN
  SAMPLE (90)

In [ ]:
%%sql -r train_result
CREATE OR REPLACE SNOWFLAKE.ML.CLASSIFICATION NORTHSTAR_ML.ML.NEIGHBORHOOD_MODEL(
    INPUT_DATA => SYSTEM$REFERENCE('VIEW', 'NORTHSTAR_ML.ML.CLASSIFICATION_TRAIN_VIEW'),
    TARGET_COLNAME => 'NEIGHBORHOOD'
)

In [ ]:
%%sql -r predictions
SELECT MONTH, DAY, NEIGHBORHOOD,
  NORTHSTAR_ML.ML.NEIGHBORHOOD_MODEL!PREDICT(INPUT_DATA => {*}) AS PREDICTIONS
FROM NORTHSTAR_ML.ML.DF_CLEAN
SAMPLE (10)

In [ ]:
%%sql -r confusion_matrix
CALL NORTHSTAR_ML.ML.NEIGHBORHOOD_MODEL!SHOW_CONFUSION_MATRIX()

In [ ]:
%%sql -r global_metrics
CALL NORTHSTAR_ML.ML.NEIGHBORHOOD_MODEL!SHOW_GLOBAL_EVALUATION_METRICS()

In [ ]:
%%sql -r feature_importance
CALL NORTHSTAR_ML.ML.NEIGHBORHOOD_MODEL!SHOW_FEATURE_IMPORTANCE()